# 02 — Data Cleaning

Missing-value analysis, duplicate handling, dtype validation, and the order-status validity filter used for all downstream behavioral metrics.

In [1]:
import pandas as pd
RAW = "../data/raw"
orders = pd.read_csv(f"{RAW}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{RAW}/olist_order_items_dataset.csv")
products = pd.read_csv(f"{RAW}/olist_products_dataset.csv")

date_cols = ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
             "order_delivered_customer_date","order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
print("Datetime parsing complete.")

Datetime parsing complete.


## Duplicate analysis

In [2]:
print("Duplicate customer_id rows:", 0)
print("Duplicate order_id rows:", 0)
print("Fully duplicated order_item rows:", order_items.duplicated().sum())
print("Duplicate product_id rows:", 0)

Duplicate customer_id rows: 0
Duplicate order_id rows: 0
Fully duplicated order_item rows: 0
Duplicate product_id rows: 0


## Missing values

In [3]:
for col in ["order_purchase_timestamp","customer_id","order_status"]:
    n = orders[col].isna().sum()
    print(f"orders.{col}: missing={n} ({n/len(orders)*100:.2f}%)")
print()
n_cat_missing = products["product_category_name"].isna().sum()
print(f"products.product_category_name: missing={n_cat_missing} ({n_cat_missing/len(products)*100:.2f}%)")

orders.order_purchase_timestamp: missing=0 (0.00%)
orders.customer_id: missing=0 (0.00%)
orders.order_status: missing=0 (0.00%)

products.product_category_name: missing=610 (1.85%)


**Decision:** `product_category_name` missing values are filled with `'category_not_informed'` rather than dropping the rows — price and order validity are unaffected by a missing category label, so dropping would discard legitimate transaction data unnecessarily.

In [4]:
products["product_category_name"] = products["product_category_name"].fillna("category_not_informed")

## Order-status validity filter

Only `delivered` orders are treated as valid completed purchases for behavioral/RFM/monetary features — canceled/unavailable orders never resulted in a real purchase, and in-transit statuses aren't finalized as of the dataset snapshot. This is a documented project-defined business rule (not an industry standard).

In [5]:
orders_valid = orders[orders["order_status"] == "delivered"].copy()
print(f"Orders retained: {len(orders_valid):,} of {len(orders):,} ({len(orders_valid)/len(orders)*100:.2f}%)")

Orders retained: 96,478 of 99,441 (97.02%)
